In [1]:
import os
import sys
sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from app.retrieval.bm25 import BM25Retriever
from app.retrieval.faiss_retriever import FaissRetriever
from app.retrieval.hybrid import HybridRetriever

In [3]:
df = pd.read_csv("../data/processed/catalog_processed.csv")
embeddings = np.load(
    "../data/embeddings/catalog_embeddings.npy"
)

In [4]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
bm25 = BM25Retriever(df)
faiss = FaissRetriever(
    df,
    embeddings
)
hybrid = HybridRetriever(
    bm25,
    faiss
)

In [6]:
query = "graduate software engineer"
query_embedding = embedder.encode([query])
results = hybrid.search(
    query=query,
    query_embedding=query_embedding,
    top_k=5
)

In [7]:
for result in results:
    print("=" * 80)
    print(result["name"])
    print(result["assessment_type"])
    print(result["job_level"])
    print(result["duration"])

Agile Software Development
Knowledge & Skills
Graduate
7 minutes
Graduate Scenarios Narrative Report
Biodata & Situational Judgment
Mid-Professional, Professional Individual Contributor, Manager, Graduate
nan
Graduate Scenarios
Biodata & Situational Judgment
Manager, Mid-Professional, Professional Individual Contributor, Graduate
Untimed
Software Business Analysis
Knowledge & Skills
Mid-Professional, Professional Individual Contributor
30 minutes
Graduate Scenarios Profile Report
Biodata & Situational Judgment
Graduate, Manager, Mid-Professional, Professional Individual Contributor
nan


In [9]:
from app.conversation.recommender import AssessmentRecommender

In [10]:
llm = AssessmentRecommender()

In [11]:
assessment_text = ""
for result in results:
    assessment_text += f"""
Name: {result['name']}
Type: {result['assessment_type']}
Job Level: {result['job_level']}
Duration: {result['duration']}
"""

In [12]:
response = llm.recommend(
    query=query,
    assessments=assessment_text
)

In [13]:
print(response)

Here's a brief explanation of each assessment and its relevance to the graduate software engineer:

1. **Agile Software Development**: This assessment evaluates the candidate's knowledge and skills in Agile methodologies, essential for any software development role. As a graduate, it assesses their understanding of Agile principles and practices.

2. **Graduate Scenarios Narrative Report**: This assessment tests the candidate's ability to apply their knowledge and experience to real-world scenarios, showcasing their problem-solving skills and maturity level. It helps predict their performance in a team setting.

3. **Graduate Scenarios**: Similar to the previous one, this untimed assessment evaluates the candidate's situational judgment and decision-making skills. It assesses their ability to handle complex situations and make sound judgments.

4. **Software Business Analysis**: As a software engineer, business analysis is an essential skillset. This 30-minute assessment tests the cand